# Held-out evaluation

Evaluates finished runs (`global_steps == 100000`) for every checkpoint key in `CONDITIONS`.

**This notebook:** ArcFace on OSNet and ConvNeXt-S, three seeds each.
The strings in `CONDITIONS` are the checkpoint folders already stored in the bucket, not the loss names.

| Family | Status |
|---|---|
| Strong Baseline, all eight objectives; OSNet and ConvNeXt-S with Circle-softmax + triplet | Done — in the paper |
| OSNet and ConvNeXt-S with ArcFace | Done — in the paper |

Re-run the **repo clone/fetch** cell so Kaggle gets latest `main`.

[`a7medlabib/reid-unseen-evaluation-targets-v1`](https://www.kaggle.com/datasets/a7medlabib/reid-unseen-evaluation-targets-v1).

**Flow**
1. Scan bucket folders and mark runs **[PASS]** when `global_steps == 100000`.
2. Verify held-out `.reidraw` targets used in `DATASETS` (TeamRecorded skipped).
3. For each finished run: download `model_best.pth`, verify SHA-256, evaluate on the four image benchmarks below.
4. Write per-run JSON/MD plus `sweep_summary.json` under `/kaggle/working/paper_eval/`.

**Outputs:** Rank-1, Rank-5, mAP, CMC 1–20 per dataset; macro mean over the four evaluated benchmarks.

MARS is omitted (tracklet protocol / positives mismatch on the unseen bundle).

**Attach:** targets dataset; secrets `GITHUB_TOKEN`, `HF_TOKEN`; 1× T4 / P100. Repo branch **`main`**.


In [ ]:
# ---- edit these ----
CONDITIONS = ("OSNet_B2", "ConvNeXtS_B2")  # checkpoint keys: ArcFace on OSNet and ConvNeXt-S
SKIP_EXISTING = True         # skip download+eval when results JSON already exists
SWEEP_TAG = "backbone_arcface_heldout"
EVAL_BATCH_SIZE = 512
CHECKPOINT = "best"          # "best" -> model_best.pth | "final" -> 50-epoch slot
DATASETS = ["market1501", "msmt17", "dukemtmc", "occludedduke"]  # MARS omitted
RESEARCH_BRANCH = "main"
HF_BUCKET_ID = "reid-research-hub/reid-loss-study-live"
BUCKET_FOLDERS = ["v3_50epoch_28c68a1", "v3_50epoch_b69c47a", "v3_50epoch_56923ec", "v3_50epoch_3caf586"]
TARGET_GLOBAL_STEPS = 50 * 2000
TOTAL_RUNS = 6               # ArcFace on OSNet and ConvNeXt-S, 3 seeds each
REPO_URL = "https://github.com/BASSAT-BASSAT/Person-Re-ID-Foundational-model.git"
# --------------------

import hashlib
import json
import subprocess
import sys
import time
from pathlib import Path

import torch
from kaggle_secrets import UserSecretsClient

assert CHECKPOINT in ("best", "final"), CHECKPOINT
ROOT = Path("/kaggle/working/Person-Re-ID-Foundational-model-eval")
WORK_ROOT = Path("/kaggle/working/paper_eval")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_CANDIDATES = [
    Path("/kaggle/input/reid-unseen-evaluation-targets-v1"),
    Path("/kaggle/input/datasets/a7medlabib/reid-unseen-evaluation-targets-v1"),
    Path("/kaggle/input/a7medlabib/reid-unseen-evaluation-targets-v1"),
]
TARGET_ROOT = next((p for p in TARGET_CANDIDATES if (p / "reid_targets_index.json").is_file()), None)
assert TARGET_ROOT is not None, "Attach dataset a7medlabib/reid-unseen-evaluation-targets-v1"

assert torch.cuda.is_available(), "Enable one GPU"
secret_client = UserSecretsClient()
GITHUB_TOKEN = secret_client.get_secret("GITHUB_TOKEN")
HF_TOKEN = secret_client.get_secret("HF_TOKEN")
assert GITHUB_TOKEN and HF_TOKEN, "Enable GITHUB_TOKEN and HF_TOKEN in Kaggle Secrets"
TIMER = {}
print(f"[PASS] GPU {torch.cuda.get_device_name(0)} | sweep {SWEEP_TAG} | targets {TARGET_ROOT}")


In [ ]:
t0 = time.perf_counter()
if not ROOT.is_dir():
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.check_call(["git", "clone", "--quiet", clone_url, str(ROOT)])
subprocess.check_call(["git", "fetch", "--quiet", "origin", RESEARCH_BRANCH], cwd=ROOT)
subprocess.check_call(
    ["git", "checkout", "--quiet", "-B", RESEARCH_BRANCH, f"origin/{RESEARCH_BRANCH}"],
    cwd=ROOT,
)
sys.path.insert(0, str(ROOT))
rev = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
subprocess.check_call([sys.executable, str(ROOT / "scripts/setup_fastreid.py"), "--cython"], cwd=ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]", "huggingface_hub>=1.7.1,<2"], cwd=ROOT)
assert (ROOT / "scripts/evaluate_unseen_reidraw.py").is_file(), f"Branch {RESEARCH_BRANCH} missing eval scripts"
from reidfm.experiments.loss_study import BACKBONE_STUDY_CONDITIONS, expand_loss_study_run, load_backbone_study_manifest
for _cond in CONDITIONS:
    assert _cond in BACKBONE_STUDY_CONDITIONS, f"Pull latest main — {_cond} missing from backbone_study.yaml"
    _run = expand_loss_study_run(_cond, 0, manifest=load_backbone_study_manifest(), output_root="outputs/backbone_study")
    _opts = dict(zip(_run.opts[::2], _run.opts[1::2]))
    assert _opts["MODEL.HEADS.CLS_LAYER"] == "ArcSoftmax", (_cond, _opts["MODEL.HEADS.CLS_LAYER"])
TIMER["setup_s"] = time.perf_counter() - t0
print(f"[PASS] Repo origin/{RESEARCH_BRANCH} @ {rev[:12]} ({TIMER["setup_s"]:.0f}s); {CONDITIONS} resolve to ArcFace")


## 1. Scan bucket — which runs finished 50 epochs?

`v3_50epoch_56923ec` is stale for most runs. Only Circle-softmax + triplet, ArcFace, and ArcFace + triplet on the Strong Baseline are accepted from that prefix.


In [ ]:
import tempfile
from huggingface_hub import download_bucket_files, list_bucket_tree

UNCHANGED_SINCE_56923EC = {"R0", "B2", "B3"}


def bucket_files(prefix):
    return {
        item.path for item in list_bucket_tree(HF_BUCKET_ID, prefix=prefix, recursive=True, token=HF_TOKEN)
        if getattr(item, "type", "file") == "file"
    }


def read_bucket_json(key):
    with tempfile.TemporaryDirectory() as td:
        dst = Path(td) / "doc.json"
        download_bucket_files(HF_BUCKET_ID, files=[(key, dst)], raise_on_missing_files=True, token=HF_TOKEN)
        return json.loads(dst.read_text(encoding="utf-8"))


runs = []
for folder in BUCKET_FOLDERS:
    keys = bucket_files(folder)
    for key in sorted(k for k in keys if k.endswith("/latest.json")):
        parts = key.split("/")
        if len(parts) != 4 or not parts[2].startswith("seed"):
            continue
        _, condition, seed_dir, _ = parts
        prefix = "/".join(parts[:3])
        doc = read_bucket_json(key)
        steps = int(doc.get("global_steps", -1))
        has_best = f"{prefix}/model_best.pth" in keys and f"{prefix}/best.json" in keys
        if steps < TARGET_GLOBAL_STEPS:
            status = "TRAINING"
        elif folder.endswith("56923ec") and condition not in UNCHANGED_SINCE_56923EC:
            status = "STALE"
        elif CHECKPOINT == "best" and not has_best:
            status = "NO_BEST"
        else:
            status = "PASS"
        runs.append({
            "folder": folder, "condition": condition, "seed": int(seed_dir[4:]), "prefix": prefix,
            "global_steps": steps, "epoch": doc.get("epoch"), "status": status, "latest": doc,
            "research_commit": str(doc.get("research_commit", ""))[:12],
        })

print(f"{'folder':<20} {'cond':<12} {'seed':<4} {'steps':>7} {'epoch':>5}  {'commit':<12} status")
for r in runs:
    print(f"{r['folder']:<20} {r['condition']:<12} {r['seed']:<4} {r['global_steps']:>7} "
          f"{str(r['epoch']):>5}  {r['research_commit']:<12} [{r['status']}]")
finished = [r for r in runs if r["status"] == "PASS"]
print(f"\n{len(finished)}/{len(runs)} runs finished all 50 epochs ({TARGET_GLOBAL_STEPS} steps)")
assert finished, "No run has reached 50 epochs yet"

to_eval = [r for r in finished if r["condition"] in CONDITIONS]
assert to_eval, f"No PASS runs for CONDITIONS={CONDITIONS} (see table above)"
print(f"Queued for held-out eval: {len(to_eval)} run(s) (expect up to {TOTAL_RUNS})")
from collections import Counter
by_cond = Counter(r["condition"] for r in to_eval)
for c in CONDITIONS:
    n = by_cond.get(c, 0)
    mark = "OK" if n == 3 else f"ONLY {n}/3 seeds"
    print(f"  {c:<12} {mark}")


## 1b. Internal-validation ranking (checkpoint-selection metric)

Reads `validation_metric` from every finished run's `best.json` (composite internal validation, no public benchmark involved) and ranks the eight SBS candidates. Paste the printed table back so the paper can state which objective internal validation selects.


In [ ]:
import statistics

SBS_CANDIDATES = ("B0", "B1", "B2", "B3", "B4", "B5", "B6", "B7")
val_rows = {}
for r in runs:
    if r["status"] != "PASS":
        continue
    try:
        meta = read_bucket_json(f"{r['prefix']}/best.json")
    except Exception as exc:
        print(f"[WARN] {r['prefix']}: {exc}")
        continue
    val_rows.setdefault(r["condition"], []).append(
        (r["seed"], float(meta.get("validation_metric", float("nan"))), int(meta.get("best_epoch_index", -1)))
    )

print(f"{'cond':<14} {'n':>2} {'val mean':>9} {'std':>6}  per-seed (seed:metric@epoch)")
summary_val = {}
for cond in sorted(val_rows):
    rows = sorted(val_rows[cond])
    vals = [v for _, v, _ in rows]
    mean = statistics.mean(vals)
    std = statistics.stdev(vals) if len(vals) > 1 else 0.0
    summary_val[cond] = {"mean": mean, "std": std, "runs": rows}
    detail = "  ".join(f"{s}:{v:.2f}@{e}" for s, v, e in rows)
    print(f"{cond:<14} {len(vals):>2} {mean:>9.2f} {std:>6.2f}  {detail}")

ranked = sorted((c for c in summary_val if c in SBS_CANDIDATES), key=lambda c: -summary_val[c]["mean"])
print("\nSBS candidates ranked by internal validation:", " > ".join(ranked))
(WORK_ROOT / "internal_validation.json").write_text(json.dumps(summary_val, indent=2), encoding="utf-8")


## 2. Verify target bundle


In [ ]:
from reidfm.data.unseen_targets import iter_target_records, load_target_specs

specs = load_target_specs(TARGET_ROOT)
for key in DATASETS:
    records = list(iter_target_records(specs[key], TARGET_ROOT))
    n_query = sum(r.get("retrieval_role") == "query" for r in records)
    assert n_query and len(records) > n_query, f"{key}: empty query or gallery"
    print(f"[PASS] {key:<13} query {n_query:>6}  gallery {len(records) - n_query:>7}")


## 3. Download + evaluate all queued runs

Resumable: set `SKIP_EXISTING = True` to continue after a Kaggle timeout.


In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        while block := stream.read(chunk):
            h.update(block)
    return h.hexdigest()


def ensure_checkpoint(run):
    work = WORK_ROOT / SWEEP_TAG / f"{run['condition']}_seed{run['seed']}"
    work.mkdir(parents=True, exist_ok=True)
    if CHECKPOINT == "best":
        meta = read_bucket_json(f"{run['prefix']}/best.json")
        remote = f"{run['prefix']}/model_best.pth"
        local = work / "model_best.pth"
    else:
        meta = run["latest"]
        remote = meta["remote_file"]
        local = work / meta["filename"]
    if not (local.is_file() and local.stat().st_size == meta.get("size")):
        t0 = time.perf_counter()
        download_bucket_files(HF_BUCKET_ID, files=[(remote, local)], raise_on_missing_files=True, token=HF_TOKEN)
        dl = time.perf_counter() - t0
    else:
        dl = 0.0
    assert local.stat().st_size == meta["size"], "size mismatch"
    assert sha256_file(local) == meta["sha256"], "sha256 mismatch"
    return work, local, meta, dl


SWEEP_DIR = WORK_ROOT / SWEEP_TAG
SWEEP_DIR.mkdir(parents=True, exist_ok=True)
sweep_rows = []
eval_script = ROOT / "scripts/evaluate_unseen_reidraw.py"

for idx, run in enumerate(to_eval, start=1):
    cond, seed = run["condition"], run["seed"]
    work = SWEEP_DIR / f"{cond}_seed{seed}"
    name = f"{cond}_seed{seed}_{CHECKPOINT}"
    result_json = work / "results" / f"{name}.json"

    if SKIP_EXISTING and result_json.is_file():
        report = json.loads(result_json.read_text(encoding="utf-8"))
        macro = report["paper"]["macro_public"]
        sweep_rows.append({
            "condition": cond, "seed": seed, "status": "skipped",
            "Rank-1": macro["Rank-1"], "Rank-5": macro["Rank-5"], "mAP": macro["mAP"],
            "folder": run["folder"], "result_json": str(result_json),
        })
        print(f"[{idx}/{len(to_eval)}] SKIP {cond} seed {seed} (existing results)")
        continue

    work, local_ckpt, meta, dl_s = ensure_checkpoint(run)
    print(f"[{idx}/{len(to_eval)}] {cond} seed {seed} | downloaded {dl_s:.1f}s | {HF_BUCKET_ID}/{run['prefix']}")

    cmd = [
        sys.executable,
        str(eval_script),
        "--checkpoint", str(local_ckpt),
        "--condition", cond,
        "--seed", str(seed),
        "--target-root", str(TARGET_ROOT),
        "--datasets", ",".join(DATASETS),
        "--name", name,
        "--out", str(work / "results"),
        "--device", "cuda",
        "--batch-size", str(EVAL_BATCH_SIZE),
        "--workers", "4",
    ]
    t0 = time.perf_counter()
    subprocess.check_call(cmd, cwd=ROOT)
    eval_s = time.perf_counter() - t0

    report = json.loads(result_json.read_text(encoding="utf-8"))
    macro = report["paper"]["macro_public"]
    sweep_rows.append({
        "condition": cond, "seed": seed, "status": "evaluated",
        "download_s": round(dl_s, 1), "eval_s": round(eval_s, 1),
        "Rank-1": macro["Rank-1"], "Rank-5": macro["Rank-5"], "mAP": macro["mAP"],
        "folder": run["folder"], "research_commit": run["research_commit"],
        "result_json": str(result_json),
    })
    print(f"       eval {eval_s / 60:.1f} min | macro R1={macro['Rank-1']:.2f}% mAP={macro['mAP']:.2f}%")

summary = {
    "sweep_tag": SWEEP_TAG,
    "checkpoint": CHECKPOINT,
    "target_root": str(TARGET_ROOT),
    "repo_rev": rev,
    "runs": sweep_rows,
}
summary_path = SWEEP_DIR / "sweep_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\n[PASS] {len(sweep_rows)} run(s) in sweep -> {summary_path}")


## 4. Sweep summary


In [ ]:
print(f"{'cond':<12} {'seed':<4} {'status':<10} {'R1':>7} {'R5':>7} {'mAP':>7}  folder")
for row in sorted(sweep_rows, key=lambda r: (r["condition"], r["seed"])):
    print(f"{row['condition']:<12} {row['seed']:<4} {row['status']:<10} "
          f"{row['Rank-1']:>7.2f} {row['Rank-5']:>7.2f} {row['mAP']:>7.2f}  {row['folder']}")

evaluated = [r for r in sweep_rows if r["status"] == "evaluated"]
if evaluated:
    avg_eval = sum(r.get("eval_s", 0) for r in evaluated) / len(evaluated)
    print(f"\nMean eval time (this session): {avg_eval / 60:.1f} min/run")
    print(f"Setup once: {TIMER.get("setup_s", 0) / 60:.1f} min")
